In [1]:
# Install MLFlow
!pip install mlflow scikit-learn pandas numpy -q

import mlflow
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

mlflow.set_experiment("SkillBridge-Skill-Matching")

# Generate skill matching data
np.random.seed(42)
n = 500
data = {
    'user_rating': np.random.uniform(1, 5, n).round(1),
    'total_sessions': np.random.randint(0, 100, n),
    'skills_count': np.random.randint(1, 8, n),
    'success_rate': np.random.uniform(60, 100, n).round(1),
    'is_verified': np.random.randint(0, 2, n),
    'hourly_rate': np.random.choice([0, 20, 30, 40, 50], n),
    'days_active': np.random.randint(1, 365, n),
    'match_score': np.random.uniform(0, 1, n).round(2)
}
df = pd.DataFrame(data)
df['good_match'] = ((df['user_rating'] >= 4.0) & (df['success_rate'] >= 80) & (df['match_score'] >= 0.6)).astype(int)

X = df.drop('good_match', axis=1)
y = df['good_match']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset: {n} records | Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Good matches: {y.sum()} ({y.mean()*100:.1f}%)")

# Experiment 1: Random Forest
with mlflow.start_run(run_name="RandomForest"):
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("accuracy", round(acc, 4))
    mlflow.log_metric("precision", round(prec, 4))
    mlflow.log_metric("recall", round(rec, 4))
    mlflow.log_metric("f1_score", round(f1, 4))

    print(f"\nRandom Forest: Accuracy={acc:.4f} F1={f1:.4f}")

# Experiment 2: Logistic Regression
with mlflow.start_run(run_name="LogisticRegression"):
    model2 = LogisticRegression(max_iter=1000, random_state=42)
    model2.fit(X_train, y_train)
    y_pred2 = model2.predict(X_test)

    acc2 = accuracy_score(y_test, y_pred2)
    prec2 = precision_score(y_test, y_pred2, zero_division=0)
    rec2 = recall_score(y_test, y_pred2, zero_division=0)
    f12 = f1_score(y_test, y_pred2, zero_division=0)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("C", 1.0)
    mlflow.log_metric("accuracy", round(acc2, 4))
    mlflow.log_metric("precision", round(prec2, 4))
    mlflow.log_metric("recall", round(rec2, 4))
    mlflow.log_metric("f1_score", round(f12, 4))

    print(f"Logistic Regression: Accuracy={acc2:.4f} F1={f12:.4f}")

# Experiment 3: Tuned Random Forest
with mlflow.start_run(run_name="TunedRandomForest"):
    model3 = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
    model3.fit(X_train, y_train)
    y_pred3 = model3.predict(X_test)

    acc3 = accuracy_score(y_test, y_pred3)
    prec3 = precision_score(y_test, y_pred3, zero_division=0)
    rec3 = recall_score(y_test, y_pred3, zero_division=0)
    f13 = f1_score(y_test, y_pred3, zero_division=0)

    mlflow.log_param("model", "TunedRandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_metric("accuracy", round(acc3, 4))
    mlflow.log_metric("precision", round(prec3, 4))
    mlflow.log_metric("recall", round(rec3, 4))
    mlflow.log_metric("f1_score", round(f13, 4))

    print(f"Tuned Random Forest: Accuracy={acc3:.4f} F1={f13:.4f}")

print("\n✅ All 3 experiments logged to MLFlow!")
print(f"Best accuracy: {max(acc, acc2, acc3):.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.

2026/04/10 14:14:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/10 14:14:55 INFO mlflow.store.db.utils: Updating database tables
2026/04/10 14:14:57 INFO mlflow.tracking.fluent: Experiment with name 'SkillBridge-Skill-Matching' does not exist. Creating a new experiment.


Dataset: 500 records | Train: 400 | Test: 100
Good matches: 23 (4.6%)

Random Forest: Accuracy=0.9600 F1=0.3333
Logistic Regression: Accuracy=0.9500 F1=0.0000
Tuned Random Forest: Accuracy=0.9700 F1=0.5714

✅ All 3 experiments logged to MLFlow!
Best accuracy: 0.9700
